# async-batch-llm: credential-free quickstart

This notebook uses a local fake async client: no API key and no model network call are required. It demonstrates progress, retries, terminal failures, summaries, checkpoint/replay, and cleanup.

In [ ]:
%pip install -q 'async-batch-llm[progress]'

In [ ]:
import asyncio
import tempfile
from pathlib import Path

from async_batch_llm import (
    ArtifactIdentity,
    CallableStrategy,
    CallOutcome,
    JsonlArtifactStore,
    ProcessorConfig,
    ResumePolicy,
    RetryConfig,
    process_prompts,
)

In [ ]:
class RetryableValidationError(RuntimeError):
    pass


class FakeClient:
    def __init__(self):
        self.calls = 0
        self.calls_by_prompt = {}

    async def generate(self, prompt, *, feedback, timeout):
        self.calls += 1
        call = self.calls_by_prompt.get(prompt, 0) + 1
        self.calls_by_prompt[prompt] = call
        await asyncio.sleep(min(0.02, timeout / 2))
        if prompt == "retry-me" and call == 1:
            raise RetryableValidationError("return a non-empty label")
        if prompt == "always-fails":
            raise RuntimeError("local demo failure")
        suffix = f" (recovered: {feedback})" if feedback else ""
        return f"label:{prompt}{suffix}"


def strategy_for(client):
    async def invoke(prompt, *, attempt, timeout, state):
        feedback = state.get("feedback") if state else None
        output = await client.generate(prompt, feedback=feedback, timeout=timeout)
        return CallOutcome(output, {"input_tokens": 2, "output_tokens": 1})

    def on_error(error, attempt, state):
        if state is not None and isinstance(error, RetryableValidationError):
            state.set("feedback", str(error))

    return CallableStrategy(
        invoke,
        on_error=on_error,
        identity=ArtifactIdentity(provider="local-demo", model="labels-v1"),
    )

In [ ]:
async def demo():
    config = ProcessorConfig(
        concurrency=3,
        attempt_timeout=1,
        retry=RetryConfig(max_attempts=2, initial_wait=0.01, max_wait=0.01),
    )
    prompts = ["alpha", "retry-me", "always-fails"]
    with tempfile.TemporaryDirectory(prefix="abl-colab-") as directory:
        store = JsonlArtifactStore(Path(directory) / "run.jsonl")
        first_client = FakeClient()
        first = await process_prompts(
            strategy_for(first_client),
            prompts,
            config=config,
            progress=True,
            artifact_store=store,
            resume=ResumePolicy.REUSE_ALL,
        )
        print(first.summary())
        for result in first.results:
            print(result.item_id, result.success, result.output or result.error_category)

        resumed_client = FakeClient()
        resumed = await process_prompts(
            strategy_for(resumed_client),
            prompts,
            config=config,
            progress=True,
            artifact_store=JsonlArtifactStore(Path(directory) / "run.jsonl"),
            resume=ResumePolicy.REUSE_ALL,
        )
        print("resume provider calls:", resumed_client.calls)
        print("replayed records:", sum(r.replayed_from_artifact for r in resumed.results))


await demo()

## Optional: replace the fake with a provider

Install the matching provider extra and set its documented environment variable. This cell is disabled so the default notebook path never makes a model call.

In [ ]:
if False:  # opt in only after setting OPENAI_API_KEY
    from async_batch_llm import llm

    real_strategy = llm("openai:gpt-4o-mini")